# Step 12 · Serve with FastAPI, call from LangChain

An OpenAI-compatible endpoint for your models — usable from curl, LangChain, Flowise.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
import time, httpx
from lab import services
services.start_api()
for _ in range(60):
    if services.api_running(): break
    time.sleep(1)
API = f"http://localhost:{services.API_PORT}"
print("API running:", services.api_running(), API)

In [ ]:
MODEL = latest("sft") or BASE
r = httpx.post(f"{API}/v1/chat/completions", timeout=300,
               json={"model": MODEL, "messages": [{"role": "user", "content": "Can I give a 20% discount?"}], "max_tokens": 80})
r.json()["choices"][0]["message"]["content"]

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
llm = ChatOpenAI(base_url=f"{API}/v1", api_key="not-needed", model=MODEL, max_tokens=80)
(ChatPromptTemplate.from_template("Answer for store staff: {q}") | llm).invoke({"q": "Do we price match marketplaces?"}).content

**Flowise** (Docker): `services.start_flowise()`, then in a chatflow use *ChatOpenAI Custom* with BasePath `http://host.docker.internal:8811/v1`.

In [ ]:
services.stop("api")   # free GPU memory